
# Task 1 - Part 1: Exploratory Data Analysis (EDA)
## Barcelona Traffic Accidents, 2019–2025

This notebook performs the bottom-up exploratory study requested in the assignment:

- describe the fields and their domains;
- inspect distributions, relationships, missing data and incoherences;
- identify unusual values/outliers;
- create exploratory charts;
- decide which fields and relationships are most useful for the final visualization.



## 1. Load and inspect the data

The dataset contains accident records for Barcelona from 2019 to 2025. The original 29 fields are kept unchanged for analysis; `fecha_dt` is added only as a parsed date helper.


In [ ]:
from IPython.display import HTML, display

display(HTML("""
<style>
div.dataframe th,
div.dataframe td {
    text-align: left !important;
}
table.dataframe th,
table.dataframe td {
    text-align: left !important;
}
</style>
"""))

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("accidents_bcn_2019_2025 (1).csv")
df["fecha_dt"] = pd.to_datetime(df["fecha"])

print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))


## 2. Field inventory and data dictionary

In [ ]:

# The completed field inventory is provided in the next output.
field_info = [
    ("numero_expedient","string / identifier","Other (record identifier)","2019S000001 … 2025S…","accident record ID","exact string","not an analytical measure"),
    ("codi_districte","numeric (float)","geographical","1–10; 232 missing","district code","integer code","source dependent"),
    ("nom_districte","categorical string","geographical","10 Barcelona districts; 232 missing","district name","categorical","source dependent"),
    ("nom_barri","categorical string","geographical","74 neighbourhoods; 232 missing","neighbourhood name","categorical","source dependent"),
    ("nom_carrer","categorical string","geographical","6,030 distinct names; 1 missing","street / location text","categorical","source dependent"),
    ("nk_any","integer","temporal","2019–2025","year","1 year","annual/source update"),
    ("mes_any","integer","temporal","1–12","month number","1 month","annual/source update"),
    ("nom_mes","categorical string","temporal","12 Catalan month names","month name","categorical","annual/source update"),
    ("dia_mes","integer","temporal","1–31","day of month","1 day","daily/source update"),
    ("hora_dia","integer","temporal","0–23","hour of day","1 hour","hourly granularity"),
    ("descripcio_dia_setmana","categorical string","temporal","7 weekdays","weekday","categorical","daily"),
    ("descripcio_torn","categorical string","temporal","Matí / Tarda / Nit","time shift","categorical","hour-derived"),
    ("descripcio_causa_vianant","categorical string","Other","8 categories","pedestrian-cause description","categorical","source dependent"),
    ("numero_morts","integer count","Other","0–1","deaths","persons","event record"),
    ("numero_lesionats_lleus","integer count","Other","0–49","minor injuries","persons","event record"),
    ("numero_lesionats_greus","integer count","Other","0–6","serious injuries","persons","event record"),
    ("numero_victimes","integer count","Other","0–53","total victims","persons","event record"),
    ("numero_vehicles_implicats","integer count","Other","0–18","vehicles involved","vehicles","event record"),
    ("longitud","numeric float","geographical","2.0580–2.2237","longitude","decimal degrees","location precision"),
    ("latitud","numeric float","geographical","41.3227–41.4675","latitude","decimal degrees","location precision"),
    ("fichero_origen","categorical string","Other","7 yearly source files","source file","categorical","yearly source"),
    ("es_causa_vianant","boolean","Other","True / False","pedestrian cause flag","boolean","source/derived"),
    ("fecha","date string","temporal","2019-01-01–2025-12-31","calendar date","1 day","daily"),
    ("dia_semana","categorical string","temporal","7 English weekday names","weekday","categorical","derived"),
    ("fin_de_semana","categorical string","temporal","Laborable / Fin de semana","weekend status","categorical","derived"),
    ("franja_horaria","categorical string","temporal","4 six-hour bands","time band","categorical","derived"),
    ("total_lesionados","integer count","Other","0–53","non-fatal injured people","persons","derived"),
    ("gravedad","categorical string","Other","Leve / Grave / Mortal","accident severity","categorical","derived"),
    ("accidente_con_victimas","boolean","Other","True / False","whether victims > 0","boolean","derived"),
]

dictionary_df = pd.DataFrame(field_info, columns=[
    "Field name","Data type + subtype","Kind","Domain / value range",
    "Unit / meaning","Required precision","Update frequency / note"
])
display(dictionary_df)


## 3. Missing data, duplicates and incoherences

In [ ]:

missing = pd.DataFrame({
    "field": df.columns,
    "missing_n": df.isna().sum().values,
})
missing["missing_pct"] = (missing["missing_n"] / len(df) * 100).round(3)
display(missing[missing["missing_n"] > 0].sort_values("missing_n", ascending=False))

print("Full duplicate rows:", df.duplicated().sum())
print("Repeated expedition identifiers:", df.duplicated("numero_expedient").sum())

checks = {
    "victims = minor + serious + deaths": (
        df["numero_victimes"] == df["numero_lesionats_lleus"] +
        df["numero_lesionats_greus"] + df["numero_morts"]
    ).all(),
    "total_lesionados = minor + serious": (
        df["total_lesionados"] == df["numero_lesionats_lleus"] +
        df["numero_lesionats_greus"]
    ).all(),
    "Mortal iff deaths > 0": (
        (df["numero_morts"] > 0) == (df["gravedad"] == "Mortal")
    ).all(),
    "victim flag iff victims > 0": (
        df["accidente_con_victimas"] == (df["numero_victimes"] > 0)
    ).all(),
    "year matches fecha": (df["nk_any"] == df["fecha_dt"].dt.year).all(),
    "month matches fecha": (df["mes_any"] == df["fecha_dt"].dt.month).all(),
    "day matches fecha": (df["dia_mes"] == df["fecha_dt"].dt.day).all(),
}
display(pd.Series(checks, name="passes"))

# The repeated expedition IDs are not exact duplicate rows.
# They differ only in pedestrian-cause category.
repeated = df[df.duplicated("numero_expedient", keep=False)].sort_values("numero_expedient")
print("Repeated IDs:", repeated["numero_expedient"].nunique())
display(repeated.groupby("numero_expedient")["descripcio_causa_vianant"].agg(list).head())


## 4. Distributions and relationships

In [ ]:

annual = df.groupby("nk_any").agg(
    accidents=("numero_expedient","size"),
    victims=("numero_victimes","sum"),
    injured=("total_lesionados","sum"),
    deaths=("numero_morts","sum"),
    serious_injuries=("numero_lesionats_greus","sum")
)
annual["serious_or_fatal_pct"] = (
    df.assign(severe=df["gravedad"].isin(["Grave","Mortal"]))
      .groupby("nk_any")["severe"].mean() * 100
)
display(annual)


In [ ]:

# Annual accident volume
plt.figure(figsize=(8,4.5))
plt.plot(annual.index, annual["accidents"], marker="o")
plt.title("Annual number of accidents")
plt.xlabel("Year")
plt.ylabel("Accidents")
plt.grid(alpha=0.25)
plt.show()


In [ ]:

# Monthly distribution
month = (
    df.groupby(["mes_any","nom_mes"]).size()
      .reset_index(name="accidents")
      .sort_values("mes_any")
)
plt.figure(figsize=(8,4.5))
plt.bar(month["nom_mes"], month["accidents"])
plt.title("Accidents by month")
plt.xlabel("Month")
plt.ylabel("Accidents")
plt.xticks(rotation=45)
plt.show()


In [ ]:

# Hour-of-day distribution
hour = df.groupby("hora_dia").size()
plt.figure(figsize=(8,4.5))
plt.bar(hour.index, hour.values)
plt.title("Accidents by hour of day")
plt.xlabel("Hour")
plt.ylabel("Accidents")
plt.xticks(range(24))
plt.show()


In [ ]:

# Weekday distribution
weekday_order = ["Dilluns","Dimarts","Dimecres","Dijous","Divendres","Dissabte","Diumenge"]
weekday = df.groupby("descripcio_dia_setmana").size().reindex(weekday_order)
plt.figure(figsize=(8,4.5))
plt.bar(weekday.index, weekday.values)
plt.title("Accidents by day of week")
plt.xlabel("Day")
plt.ylabel("Accidents")
plt.xticks(rotation=30)
plt.show()


In [ ]:

# Severity distribution
severity = df["gravedad"].value_counts().reindex(["Leve","Grave","Mortal"]).fillna(0)
plt.figure(figsize=(7,4.5))
plt.bar(severity.index, severity.values)
plt.title("Accident severity")
plt.xlabel("Severity")
plt.ylabel("Number of records")
plt.show()


In [ ]:

# District distribution
district = (
    df.groupby("nom_districte")
      .agg(accidents=("numero_expedient","size"),
           victims=("numero_victimes","sum"),
           deaths=("numero_morts","sum"))
      .sort_values("accidents", ascending=False)
)
plt.figure(figsize=(8,4.5))
plt.barh(district.index[::-1], district["accidents"].values[::-1])
plt.title("Accidents by district")
plt.xlabel("Accidents")
plt.ylabel("District")
plt.show()


In [ ]:

# Weekday x hour relationship
heat = pd.crosstab(df["descripcio_dia_setmana"], df["hora_dia"]).reindex(weekday_order)
plt.figure(figsize=(10,4.8))
plt.imshow(heat.values, aspect="auto")
plt.title("Accidents by weekday and hour")
plt.xlabel("Hour of day")
plt.ylabel("Day of week")
plt.xticks(range(24), range(24))
plt.yticks(range(7), heat.index)
plt.colorbar(label="Accidents")
plt.show()


In [ ]:

# Geographic distribution
geo = df.dropna(subset=["longitud","latitud"])
plt.figure(figsize=(7,6))
plt.scatter(geo["longitud"], geo["latitud"], s=2, alpha=0.25)
plt.title("Geographic distribution of recorded accidents")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()



## 5. Outliers and unusual values

The count variables are strongly right-skewed. The most extreme observations are:

- maximum minor injuries in one record: **49**;
- maximum total victims in one record: **53**;
- maximum vehicles involved in one record: **18**.

These are not automatically errors: they represent unusually large accident events and should remain in the dataset. They can, however, strongly influence averages, so medians/count distributions are preferable for some summaries.


In [ ]:

for c in ["numero_lesionats_lleus","numero_victimes","numero_vehicles_implicats"]:
    i = df[c].idxmax()
    print(c, "maximum:", df.loc[i, c])
    print(df.loc[i, ["numero_expedient","fecha","nom_districte","nom_carrer",c]].to_dict())
    print()



## 6. From EDA to visualization

### Main findings

1. **Accident volume changes substantially over time**, with a pronounced fall in 2020 and a partial recovery afterwards.
2. **There is a strong time-of-day pattern**: accidents are concentrated from late morning through evening and are least frequent overnight.
3. **Seasonality exists**: July is the highest month and August the lowest in the combined 2019–2025 records.
4. **Severity is highly imbalanced**, with most accidents classified as minor, but the serious/fatal share rises from 2019 to 2024–2025.
5. **Eixample has the highest absolute number of records**, but this should not be interpreted as the highest risk without exposure/population/traffic denominators.
6. **Location is suitable for a geographic visualization**, because latitude/longitude are available for 54,912 of 54,954 records.
7. The dataset has **low overall missingness**, but 232 records lack district/neighbourhood information and 42 lack coordinates.

### Fields selected for the final visualization

The final 800×450 image focuses on:

- `nk_any` - year;
- accident record count;
- `gravedad` - summarized as the share of `Grave` + `Mortal`.

This selection tells a compact story: **the number of accidents fell sharply in 2020, while the proportion of serious/fatal records increased in later years**. Time-of-day, geography and district are retained as supporting EDA rather than overloaded into the final graphic.



## 7. Final visualization

The submission image is saved separately as `task01_visualization_800x450.png` and is embedded in `task01.html`.


In [ ]:

# Recreate the final 800 x 450 visualization
fig, axes = plt.subplots(1, 2, figsize=(8, 4.5), dpi=100)

axes[0].plot(annual.index, annual["accidents"], marker="o", linewidth=2)
axes[0].set_title("Barcelona traffic accidents, 2019–2025")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Number of accidents")
axes[0].grid(alpha=0.25)
axes[0].set_xticks(annual.index)

axes[1].plot(annual.index, annual["serious_or_fatal_pct"], marker="o", linewidth=2)
axes[1].set_title("Share classified as serious or fatal")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Accidents (%)")
axes[1].grid(alpha=0.25)
axes[1].set_xticks(annual.index)

fig.suptitle("From accident volume to accident severity", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()
